In [5]:
# Cell 1 — Imports and config
import sqlite3
import yaml
from openai import OpenAI
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from config.config import OPENAI_API_KEY

NOTEGROUP_ID = 1
DB_PATH = "../DB/oedb_baseline_v3.db"
PROMPT_PATH_REFINER  = "../data/prompt_templates/refiner/abandoned_prompt_refiner_ParReducer.yaml"
SERVICE_ACCOUNT_FILE = "../config/service_account_key.json"
LLM_MODEL = "gpt-5.1"

client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://llmproxy.uva.nl/v1")

In [6]:
# Cell 2 — Inject the wrong result to test the refiner against
result_lastcall = """[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]"""

In [7]:
# Cell 3 — Load texts from Google Drive (same as ETL_baseline_v3.py)
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        SELECT note_url_QA, note_url_PARTICIPANT
        FROM notegroups
        WHERE notegroupID = ?
    """, (NOTEGROUP_ID,))
    row = cursor.fetchone()
    note_url_qa, note_url_participant = row

file_loader = GoogleDriveLoader(SERVICE_ACCOUNT_FILE)
extractor   = TextExtractor()

all_texts = {}
for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
    if not url:
        continue
    result = file_loader.load(url)
    all_texts[label] = f"[Data source: {result['name']}]\n{extractor.extract(result)}"

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


In [8]:
# Cell 4 — Run the refiner
def call_refiner(client, prompts, text_qa, text_par, result_lastcall):
    system_prompt = prompts["system"]
    user_prompt   = prompts["user"]["refine"].format(
        text_qa=text_qa,
        text_par=text_par,
        result_lastcall=result_lastcall
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        seed=42,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
    )
    return response.choices[0].message.content.strip()


prompts     = yaml.safe_load(open(PROMPT_PATH_REFINER))
text_qa     = all_texts.get("QA", "")
text_par    = all_texts.get("PARTICIPANT", "")

output = call_refiner(client, prompts, text_qa, text_par, result_lastcall)

parts = output.split("###RESULT###")
if len(parts) < 2:
    print("WARNING: ###RESULT### marker not found, keeping original result.")
    final_result = result_lastcall
else:
    result = parts[-1].strip()
    if result.lower() == "pass":
        print("Refiner returned: pass")
        final_result = result_lastcall
    else:
        print("Refiner returned a correction.")
        final_result = result

print("\n=== Final refined result ===")
print(final_result)

Refiner returned a correction.

=== Final refined result ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | null |
| Eyas | Female | Syria | B1 | 30 | Gemert | null |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | null |
| Nedal | Male | Syria | Z route, | 53 | Helmond | null |
| Hassan | Male | Syria | Z route, | 30 | Helmond | null |
| Wasim | Male | Syria | Zroute, | 53 | Helmond | null |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | F.A |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]
